<a href="https://colab.research.google.com/github/mohashafi/ProgramingforAI_FALL26/blob/main/tsp_genetic_algorith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import random
DISTANCES = {
    ('Islamabad', 'Lahore'): 375,
    ('Lahore', 'Faisalabad'): 180,
    ('Faisalabad', 'Islamabad'): 320,
    ('Islamabad', 'Peshawar'): 185,
    ('Peshawar', 'Quetta'): 850,
    ('Quetta', 'Karachi'): 680,
    ('Karachi', 'Lahore'): 1210,
    ('Islamabad', 'Karachi'): 1410,
    ('Islamabad', 'Quetta'): 950,
    ('Lahore', 'Peshawar'): 500,
    ('Lahore', 'Quetta'): 1160,
    ('Faisalabad', 'Karachi'): 1150,
    ('Faisalabad', 'Peshawar'): 440,
    ('Faisalabad', 'Quetta'): 920,
    ('Peshawar', 'Karachi'): 1530,
}

ALL_CITIES = ['Islamabad', 'Lahore', 'Karachi', 'Peshawar', 'Quetta', 'Faisalabad']

def distance_between(city1, city2):
    """Retrieves distance between two cities regardless of pair order."""
    if (city1, city2) in DISTANCES:
        return DISTANCES[(city1, city2)]
    elif (city2, city1) in DISTANCES:
        return DISTANCES[(city2, city1)]
    return float('inf')

def tour_distance(chromosome, start_city):
    """Calculates total distance of a tour including return to start city."""
    tour = [start_city] + chromosome + [start_city]
    total = 0
    for i in range(len(tour) - 1):
        total += distance_between(tour[i], tour[i + 1])
    return total

def tournament_selection(population, start_city, k=3):
    """Tournament selection mechanism to choose parents for reproduction."""
    sample = random.sample(population, k)
    sample.sort(key=lambda chrom: tour_distance(chrom, start_city))
    return list(sample[0])

def order_crossover(parent1, parent2):
    """Order Crossover (OX) to generate offspring."""
    size = len(parent1)
    child = [None] * size
    start, end = sorted(random.sample(range(size), 2))

    # Copy segment from parent 1
    child[start:end + 1] = parent1[start:end + 1]

    # Fill remaining positions from parent 2
    p2_idx = (end + 1) % size
    child_idx = (end + 1) % size

    while None in child:
        city = parent2[p2_idx]
        if city not in child:
            child[child_idx] = city
            child_idx = (child_idx + 1) % size
        p2_idx = (p2_idx + 1) % size

    return child

def mutate(chromosome, mutation_rate=0.1):
    """Swap mutation operator to introduce diversity into the population."""
    chrom = list(chromosome)
    if random.random() < mutation_rate:
        idx1, idx2 = random.sample(range(len(chrom)), 2)
        chrom[idx1], chrom[idx2] = chrom[idx2], chrom[idx1]
    return chrom

def genetic_algorithm(start_city, pop_size=50, max_gen=300, patience=60):
    # Set fixed random seed for reproducibility
    random.seed(42)

    other_cities = [c for c in ALL_CITIES if c != start_city]

    # Population Initialization: generate initial random population of tours
    population = []
    for _ in range(pop_size):
        chrom = list(other_cities)
        random.shuffle(chrom)
        population.append(chrom)

    best_chrom = None
    best_dist = float('inf')
    generations_without_improvement = 0
    stopped_generation = max_gen

    for gen in range(1, max_gen + 1):
        # Fitness Function: evaluate population based on total distance
        population.sort(key=lambda chrom: tour_distance(chrom, start_city))
        current_best_dist = tour_distance(population[0], start_city)

        if current_best_dist < best_dist:
            best_dist = current_best_dist
            best_chrom = list(population[0])
            generations_without_improvement = 0
        else:
            generations_without_improvement += 1

        # Termination Criteria: stop if fitness improvement stops
        if generations_without_improvement >= patience:
            stopped_generation = gen
            break

        # Elitism: preserve top 2 individuals into the next generation
        new_population = [list(population[0]), list(population[1])]

        while len(new_population) < pop_size:
            p1 = tournament_selection(population, start_city)
            p2 = tournament_selection(population, start_city)
            child = order_crossover(p1, p2)
            child = mutate(child, mutation_rate=0.2)
            new_population.append(child)

        population = new_population
    else:
        stopped_generation = max_gen

    return best_chrom, best_dist, stopped_generation

def main():
    print("=" * 62)
    print("COMSATS UNIVERSITY ISLAMABAD - TSP GENETIC ALGORITHM")
    print("=" * 62)
    print("\nAvailable cities:", ", ".join(ALL_CITIES))

    start_city = input("\nEnter the starting city exactly as shown above: ").strip()

    if start_city not in ALL_CITIES:
        print(f"Error: '{start_city}' is not in the list of available cities.")
        return

    best_chrom, best_dist, stopped_gen = genetic_algorithm(start_city)

    print(f"\nStopped early at generation {stopped_gen}: no improvement.")
    print("\n\nBest tour found (Output):\n")

    full_tour = [start_city] + best_chrom + [start_city]
    print(" -> ".join(full_tour))
    print(f"\nTotal travel distance (including return to start): {best_dist} km")
    print("\n\nNote: This is a heuristic optimization result.")

if __name__ == "__main__":
    main()

COMSATS UNIVERSITY ISLAMABAD - TSP GENETIC ALGORITHM

Available cities: Islamabad, Lahore, Karachi, Peshawar, Quetta, Faisalabad

Enter the starting city exactly as shown above: Faisalabad

Stopped early at generation 63: no improvement.


Best tour found (Output):

Faisalabad -> Lahore -> Islamabad -> Peshawar -> Quetta -> Karachi -> Faisalabad

Total travel distance (including return to start): 3420 km


Note: This is a heuristic optimization result.
